# Semantic Chunking + RBAC RAG in SageMaker

**Track:** Applied Agent Engineering Foundations  
**Module fit:** RAG Fundamentals & Retrieval Patterns  

## What learners will understand
- how to read policy documents and salary data from S3
- how to chunk documents into retrieval units
- how to generate embeddings with Bedrock
- how to perform retrieval without FAISS using in-memory cosine similarity
- how RBAC changes what evidence is allowed to reach the model
- how to create a retrieval audit trail that is easy to read in class

## Notebook assumptions
- runs in **SageMaker Jupyter Notebook**
- uses **Amazon Bedrock**
- can read data from **S3**
- keeps outputs in **DataFrames only**


## Classroom framing

A lot of RAG demos retrieve *relevant* context.  
Enterprise RAG must retrieve context that is both:
- relevant
- authorized

That is the point of this lab.


In [18]:

# Uncomment only if your image is missing dependencies.
# %pip install -q boto3 pandas numpy python-docx


## Step 1 — Configuration

In [19]:
from __future__ import annotations

import io
import json
import os
import re

import boto3
import numpy as np
import pandas as pd

AWS_REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")

# LLM model for answer generation and optional reranking.
BEDROCK_LLM_MODEL_ID = "amazon.nova-lite-v1:0"

# Bedrock embedding model used in this notebook.
BEDROCK_EMBEDDING_MODEL_ID = "amazon.titan-embed-text-v1"

# S3 
S3_BUCKET = "rag-demo-docs-sri"
POLICY_PREFIX = "rag-docs/"
SALARY_KEY = "small_company_20_employees_7_roles_salary.csv"


TOP_K = 4
OVERFETCH_K = 12
MAX_CHARS_PER_CHUNK = 350

from docx import Document as DocxDocument


## Step 2 — Load policy documents and salary roster from S3

This notebook reads:
- policy documents (`.docx`, `.txt`, `.md`) from S3
- one salary roster CSV from S3

For **normal policy questions**, the notebook performs RAG over the policy documents from S3.  
For **salary questions**, the notebook also applies RBAC before sending context to the LLM.


In [20]:

s3 = boto3.client("s3", region_name=AWS_REGION)
bedrock_runtime = boto3.client("bedrock-runtime", region_name=AWS_REGION)

SUPPORTED_POLICY_EXTENSIONS = (".docx", ".txt", ".md")

def read_csv_from_s3(bucket: str, key: str) -> pd.DataFrame:
    body = s3.get_object(Bucket=bucket, Key=key)["Body"].read()
    return pd.read_csv(io.BytesIO(body))

def list_policy_document_keys(bucket: str, prefix: str) -> list[str]:
    paginator = s3.get_paginator("list_objects_v2")
    keys = []
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if key.endswith("/"):
                continue
            lowered = key.lower()
            if lowered.endswith(SUPPORTED_POLICY_EXTENSIONS):
                keys.append(key)
    return sorted(keys)

def read_docx_from_s3(bucket: str, key: str) -> str:
    body = s3.get_object(Bucket=bucket, Key=key)["Body"].read()
    doc = DocxDocument(io.BytesIO(body))
    paragraphs = []
    for p in doc.paragraphs:
        text = (p.text or "").strip()
        if text:
            paragraphs.append(text)
    return "\n\n".join(paragraphs)

def read_text_like_from_s3(bucket: str, key: str) -> str:
    return s3.get_object(Bucket=bucket, Key=key)["Body"].read().decode("utf-8", errors="ignore")

def read_policy_document_from_s3(bucket: str, key: str) -> str:
    lowered = key.lower()
    if lowered.endswith(".docx"):
        return read_docx_from_s3(bucket, key)
    if lowered.endswith(".txt") or lowered.endswith(".md"):
        return read_text_like_from_s3(bucket, key)
    raise ValueError(f"Unsupported policy document type for key: {key}")

policy_keys = list_policy_document_keys(S3_BUCKET, POLICY_PREFIX)
policy_docs = [
    {"source": key, "text": read_policy_document_from_s3(S3_BUCKET, key)}
    for key in policy_keys
]

if not policy_docs:
    raise ValueError(
        f"No policy documents were found in s3://{S3_BUCKET}/{POLICY_PREFIX}. "
        "Upload at least one .docx, .txt, or .md policy document."
    )

salary_df = read_csv_from_s3(S3_BUCKET, SALARY_KEY)

print("Policy documents loaded from S3:")
display(pd.DataFrame(policy_docs)[["source"]])

print("Salary roster loaded from S3:")
display(salary_df)


Policy documents loaded from S3:


,source
0,rag-docs/HR-POL-001_Employee_Onboarding_Policy...
1,rag-docs/HR-POL-002_Equal_Opportunity_Policy.docx
2,rag-docs/HR-POL-003_Health_Benefits_Policy.docx
3,rag-docs/HR-POL-004_Training_and_Development_P...
4,rag-docs/HR-POL-005_Termination_and_Suspension...
5,rag-docs/HR-POL-006_Performance_Appraisal_Poli...
6,rag-docs/HR-POL-007_General_Work_Policy.docx
7,rag-docs/HR-POL-008_Leave_Policy.docx
8,rag-docs/HR-POL-009_Remote_Work_Policy.docx


Salary roster loaded from S3:


,employee_id,employee_name,level,role,monthly_salary_inr,annual_salary_inr
0,EMP-1001,Aarav Sharma,L1,Associate Engineer,35000,420000
1,EMP-1002,Diya Nair,L1,Associate Engineer,36000,432000
2,EMP-1003,Ishaan Gupta,L1,Associate Engineer,34000,408000
3,EMP-1004,Meera Reddy,L2,Engineer,50000,600000
4,EMP-1005,Rohan Iyer,L2,Engineer,52000,624000
5,EMP-1006,Kavya Menon,L2,Engineer,51000,612000
6,EMP-1007,Aditya Rao,L2,Engineer,53000,636000
7,EMP-1008,Sneha Kulkarni,L3,Senior Engineer,70000,840000
8,EMP-1009,Arjun Verma,L3,Senior Engineer,72000,864000
9,EMP-1010,Pooja Bhat,L3,Senior Engineer,71000,852000


## Step 3 — Build policy chunks from S3 documents and salary retrieval rows

In [21]:
def semantic_like_chunks(text: str, max_chars: int = MAX_CHARS_PER_CHUNK) -> list[str]:
    text = (text or "").strip()
    if not text:
        return []

    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    if not paragraphs:
        paragraphs = [text]

    chunks = []
    current = ""

    for para in paragraphs:
        candidate = f"{current}\n\n{para}".strip() if current else para
        if len(candidate) <= max_chars:
            current = candidate
        else:
            if current:
                chunks.append(current)
            if len(para) <= max_chars:
                current = para
            else:
                sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", para) if s.strip()]
                temp = ""
                for sent in sentences:
                    sent_candidate = f"{temp} {sent}".strip() if temp else sent
                    if len(sent_candidate) <= max_chars:
                        temp = sent_candidate
                    else:
                        if temp:
                            chunks.append(temp)
                        temp = sent
                current = temp

    if current:
        chunks.append(current)

    return chunks

policy_chunk_rows = []
for doc in policy_docs:
    chunks = semantic_like_chunks(doc["text"], max_chars=MAX_CHARS_PER_CHUNK)
    for i, ch in enumerate(chunks, start=1):
        policy_chunk_rows.append({
            "doc_type": "policy",
            "source": doc["source"],
            "doc_id": doc["source"],
            "chunk_id": f'{doc["source"]}::chunk_{i:03d}',
            "text": ch,
            "employee_id": None,
            "level_num": None
        })

salary_docs = []
for r in salary_df.itertuples():
    level_num = int(re.findall(r"\d+", r.level)[0])
    salary_docs.append({
        "doc_type": "salary",
        "source": "salary_roster.csv",
        "doc_id": r.employee_id,
        "chunk_id": f"{r.employee_id}::salary",
        "text": f"Employee ID: {r.employee_id}. Employee Name: {r.employee_name}. Level: {r.level}. Role: {r.role}. Annual Salary INR: {int(r.annual_salary_inr)}.",
        "employee_id": r.employee_id,
        "level_num": level_num
    })

corpus_df = pd.DataFrame(policy_chunk_rows + salary_docs)
print("Chunking mode: paragraph-and-sentence aware")
display(corpus_df)


Chunking mode: paragraph-and-sentence aware


,doc_type,source,doc_id,chunk_id,text,employee_id,level_num
0,policy,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,ABC Solutions Pvt. Ltd.\nDocument ID: HR-POL-0...,None,NaN
1,policy,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,"This policy establishes a standardized, legall...",None,NaN
2,policy,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,The objective is to ensure timely statutory re...,None,NaN
3,policy,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,2. Scope,None,NaN
4,policy,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,rag-docs/HR-POL-001_Employee_Onboarding_Policy...,"This policy applies to all permanent, fixed-te...",None,NaN
...,...,...,...,...,...,...,...
189,salary,salary_roster.csv,EMP-1016,EMP-1016::salary,Employee ID: EMP-1016. Employee Name: Neha Kap...,EMP-1016,5.0
190,salary,salary_roster.csv,EMP-1017,EMP-1017::salary,Employee ID: EMP-1017. Employee Name: Sanjay P...,EMP-1017,5.0
191,salary,salary_roster.csv,EMP-1018,EMP-1018::salary,Employee ID: EMP-1018. Employee Name: Lakshmi ...,EMP-1018,6.0
192,salary,salary_roster.csv,EMP-1019,EMP-1019::salary,Employee ID: EMP-1019. Employee Name: Farah Kh...,EMP-1019,6.0


## Step 4 — Create the retrieval matrix with Bedrock embeddings

In [22]:
def embed_text_bedrock(text: str, model_id: str = BEDROCK_EMBEDDING_MODEL_ID) -> list[float]:
    payload = {"inputText": text}
    response = bedrock_runtime.invoke_model(
        modelId=model_id,
        body=json.dumps(payload),
        contentType="application/json",
        accept="application/json",
    )
    body = json.loads(response["body"].read())
    if "embedding" not in body:
        raise ValueError(f"Embedding response missing 'embedding': {body}")
    return body["embedding"]

def l2_normalize(arr: np.ndarray) -> np.ndarray:
    denom = np.linalg.norm(arr, axis=1, keepdims=True)
    denom = np.where(denom == 0, 1.0, denom)
    return arr / denom

corpus_embeddings = [embed_text_bedrock(text) for text in corpus_df["text"].tolist()]
embedding_matrix = np.array(corpus_embeddings, dtype="float32")
embedding_matrix = l2_normalize(embedding_matrix)

print("Embedded rows:", len(embedding_matrix))
print("Embedding dimension:", embedding_matrix.shape[1])


Embedded rows: 194
Embedding dimension: 1536


## Step 5 — RBAC rules + retrieval audit table

Salary access rule used in this teaching notebook:
- L1–L4: can see only their own salary  
- L5–L6: can see their own salary plus salaries of people below their level  
- L7: can see all salaries

Important:
- For **salary questions**, the notebook reads the requester level directly from `salary_df`.
- For **normal policy questions**, the notebook should still work even if the requester employee ID is blank.
- Salary rows are automatically hidden for non-salary questions.

This makes the same notebook useful for both:
1. normal RAG over policy text
2. secure RAG over salary data


In [23]:
def parse_level(level: str) -> int:
    return int(re.findall(r"\d+", level)[0])

def normalize_employee_id(text: str | None) -> str | None:
    if text is None:
        return None
    text = str(text).strip().upper()
    if not text:
        return None
    if re.fullmatch(r"EMP-\d{4}", text):
        return text
    return text

def get_requester_level_from_df(requester_employee_id: str | None) -> int:
    requester_employee_id = normalize_employee_id(requester_employee_id)
    if not requester_employee_id:
        raise ValueError("Requester employee id is required for salary questions.")
    requester_row = salary_df.loc[
        salary_df["employee_id"].astype(str).str.upper() == requester_employee_id
    ]
    if len(requester_row) == 0:
        raise ValueError(
            f"Requester employee id '{requester_employee_id}' was not found in salary_df."
        )
    return parse_level(str(requester_row.iloc[0]["level"]))

def extract_requested_employee_ids(query: str, requester_employee_id: str | None = None) -> list[str]:
    """Extract explicit employee ids from the question.
    Also supports 'my salary' by mapping to the requester id.
    """
    ids = re.findall(r"EMP-\d{4}", (query or "").upper())
    ids = list(dict.fromkeys(ids))
    requester_employee_id = normalize_employee_id(requester_employee_id)
    if not ids and requester_employee_id and re.search(r"\bmy\b", (query or "").lower()):
        ids = [requester_employee_id]
    return ids

def allowed_salary_ids(requester_level: int | None, requester_employee_id: str | None) -> set[str]:
    requester_employee_id = normalize_employee_id(requester_employee_id)

    if requester_level is None:
        return set()

    if requester_level >= 7:
        return set(salary_df["employee_id"].astype(str).str.upper().tolist())

    if requester_level <= 4:
        return {requester_employee_id} if requester_employee_id else set()

    requester_row = salary_df.loc[
        salary_df["employee_id"].astype(str).str.upper() == requester_employee_id
    ]
    if len(requester_row) == 0:
        return set()

    requester_level_num = parse_level(requester_row.iloc[0]["level"])

    allowed = set(
        salary_df.loc[
            salary_df["level"].apply(parse_level) < requester_level_num,
            "employee_id"
        ].astype(str).str.upper().tolist()
    )
    if requester_employee_id:
        allowed.add(requester_employee_id)
    return allowed

def is_salary_question(q: str) -> bool:
    return bool(re.search(r"\b(salary|pay|compensation|ctc)\b", (q or "").lower()))

def build_retrieval_audit_table(
    query: str,
    requester_level: int | None,
    requester_employee_id: str | None,
    top_k: int = TOP_K
) -> tuple[pd.DataFrame, pd.DataFrame]:
    requester_employee_id = normalize_employee_id(requester_employee_id)
    query_is_salary = is_salary_question(query)
    requested_ids = extract_requested_employee_ids(query, requester_employee_id)
    allowed_ids = allowed_salary_ids(requester_level, requester_employee_id)

    q_emb = np.array([embed_text_bedrock(query)], dtype="float32")
    q_emb = l2_normalize(q_emb)
    similarities = embedding_matrix @ q_emb.T
    ranked_indices = np.argsort(-similarities[:, 0])[: min(OVERFETCH_K, len(corpus_df))]

    audit_rows = []
    for rank_position, idx_ in enumerate(ranked_indices, start=1):
        row = corpus_df.iloc[int(idx_)].to_dict()
        row_employee_id = normalize_employee_id(row["employee_id"]) if pd.notna(row["employee_id"]) and row["employee_id"] else None

        access_decision = "allowed"
        reason = "policy chunks are always allowed"

        if row["doc_type"] == "salary":
            if not query_is_salary:
                access_decision = "blocked"
                reason = "salary rows are hidden for non-salary questions"
            elif row_employee_id not in allowed_ids:
                access_decision = "blocked"
                reason = "salary row exists but requester is not authorized for this employee"
            elif requested_ids and row_employee_id in requested_ids:
                access_decision = "allowed"
                reason = "requested salary row and requester is authorized"
            else:
                access_decision = "allowed"
                reason = "salary row is within requester authorization scope"

        audit_rows.append({
            "retrieval_rank": rank_position,
            "doc_type": row["doc_type"],
            "chunk_id": row["chunk_id"],
            "employee_id": row_employee_id,
            "similarity_score": round(float(similarities[int(idx_), 0]), 4),
            "access_decision": access_decision,
            "why_this_row_is_allowed_or_blocked": reason,
            "text_preview": row["text"][:140] + ("..." if len(row["text"]) > 140 else "")
        })

    audit_df = pd.DataFrame(audit_rows)

    forced_salary_rows = pd.DataFrame()
    if query_is_salary and requested_ids:
        forced_salary_rows = corpus_df[
            (corpus_df["doc_type"] == "salary") &
            (corpus_df["employee_id"].astype(str).str.upper().isin(requested_ids)) &
            (corpus_df["employee_id"].astype(str).str.upper().isin(allowed_ids))
        ].copy()

    allowed_ranked_rows = []
    seen_chunk_ids = set()

    if len(forced_salary_rows):
        for _, row in forced_salary_rows.iterrows():
            chunk_id = row["chunk_id"]
            if chunk_id in seen_chunk_ids:
                continue
            seen_chunk_ids.add(chunk_id)
            row_dict = row.to_dict()
            row_dict["similarity_score"] = None
            row_dict["selection_reason"] = "forced into final context because it is the explicitly requested and authorized salary row"
            allowed_ranked_rows.append(row_dict)

    for idx_ in ranked_indices:
        row = corpus_df.iloc[int(idx_)].to_dict()
        row_employee_id = normalize_employee_id(row["employee_id"]) if pd.notna(row["employee_id"]) and row["employee_id"] else None

        if row["doc_type"] == "salary":
            if not query_is_salary:
                continue
            if row_employee_id not in allowed_ids:
                continue

        chunk_id = row["chunk_id"]
        if chunk_id in seen_chunk_ids:
            continue

        seen_chunk_ids.add(chunk_id)
        row["similarity_score"] = round(float(similarities[int(idx_), 0]), 4)
        row["selection_reason"] = "selected by retrieval ranking after RBAC filtering"
        allowed_ranked_rows.append(row)

        if len(allowed_ranked_rows) >= top_k:
            break

    authorized_context_df = pd.DataFrame(allowed_ranked_rows)
    return audit_df, authorized_context_df



In [24]:
# ------------------------------------------------------------------
# For salary questions, provide a requester employee ID.
# For normal policy questions, requester_employee_id can be blank.
# ------------------------------------------------------------------
requester_employee_id = "EMP-1006"
# query_to_test = "What is EMP-1015 salary?"
query_to_test = "What are the rules for remote work?"

normalized_requester_employee_id = normalize_employee_id(requester_employee_id)
query_is_salary = is_salary_question(query_to_test)

if query_is_salary:
    requester_level = get_requester_level_from_df(normalized_requester_employee_id)
    requester_level_label = f"L{requester_level}"
else:
    requester_level = None
    requester_level_label = "Not needed for non-salary question"

print(f"Requester employee id: {normalized_requester_employee_id if normalized_requester_employee_id else '(blank)'}")
print(f"Requester level (read from salary_df only when needed): {requester_level_label}")
print(f"Query to test: {query_to_test}")

retrieval_audit_df, authorized_context_df = build_retrieval_audit_table(
    query=query_to_test,
    requester_level=requester_level,
    requester_employee_id=normalized_requester_employee_id,
    top_k=TOP_K
)

print("Retrieval audit table: this shows what the retriever found and how RBAC evaluated each row.")
display(retrieval_audit_df)

print("Authorized RAG context table: these are the rows that are finally allowed to reach the model.")
display(authorized_context_df[["doc_type", "chunk_id", "employee_id", "selection_reason", "text"]])


Requester employee id: EMP-1006
Requester level (read from salary_df only when needed): Not needed for non-salary question
Query to test: What are the rules for remote work?
Retrieval audit table: this shows what the retriever found and how RBAC evaluated each row.


,retrieval_rank,doc_type,chunk_id,employee_id,similarity_score,access_decision,why_this_row_is_allowed_or_blocked,text_preview
0,1,policy,rag-docs/HR-POL-009_Remote_Work_Policy.docx::c...,None,0.7263,allowed,policy chunks are always allowed,A combination of on-site and remote work as pe...
1,2,policy,rag-docs/HR-POL-009_Remote_Work_Policy.docx::c...,None,0.7129,allowed,policy chunks are always allowed,"Employees on probation, under Performance Impr..."
2,3,policy,rag-docs/HR-POL-009_Remote_Work_Policy.docx::c...,None,0.6947,allowed,policy chunks are always allowed,10.1 Remote work approval shall be reviewed qu...
3,4,policy,rag-docs/HR-POL-009_Remote_Work_Policy.docx::c...,None,0.6828,allowed,policy chunks are always allowed,This policy applies to all confirmed employees...
4,5,policy,rag-docs/HR-POL-007_General_Work_Policy.docx::...,None,0.6740,allowed,policy chunks are always allowed,Governs disciplinary action for serious or rep...
5,6,policy,rag-docs/HR-POL-009_Remote_Work_Policy.docx::c...,None,0.6548,allowed,policy chunks are always allowed,4.2 Approval Levels:\n4.2.1 Up to 2 days per w...
6,7,policy,rag-docs/HR-POL-008_Leave_Policy.docx::chunk_014,None,0.6045,allowed,policy chunks are always allowed,Temporary work-from-home during medical recove...
7,8,policy,rag-docs/HR-POL-006_Performance_Appraisal_Poli...,None,0.5694,allowed,policy chunks are always allowed,9.3 Remote Work Eligibility: Employees with OP...
8,9,policy,rag-docs/HR-POL-007_General_Work_Policy.docx::...,None,0.5651,allowed,policy chunks are always allowed,"This policy applies to all permanent, fixed-te..."
9,10,policy,rag-docs/HR-POL-009_Remote_Work_Policy.docx::c...,None,0.5624,allowed,policy chunks are always allowed,ABC Solutions Pvt. Ltd.\n\nABC Solutions Pvt. ...


Authorized RAG context table: these are the rows that are finally allowed to reach the model.


,doc_type,chunk_id,employee_id,selection_reason,text
0,policy,rag-docs/HR-POL-009_Remote_Work_Policy.docx::c...,None,selected by retrieval ranking after RBAC filte...,A combination of on-site and remote work as pe...
1,policy,rag-docs/HR-POL-009_Remote_Work_Policy.docx::c...,None,selected by retrieval ranking after RBAC filte...,"Employees on probation, under Performance Impr..."
2,policy,rag-docs/HR-POL-009_Remote_Work_Policy.docx::c...,None,selected by retrieval ranking after RBAC filte...,10.1 Remote work approval shall be reviewed qu...
3,policy,rag-docs/HR-POL-009_Remote_Work_Policy.docx::c...,None,selected by retrieval ranking after RBAC filte...,This policy applies to all confirmed employees...


## Step 6 — LLM-generated answer over authorized context only

This step behaves like a proper **RAG answer generation** stage:
- retrieve relevant evidence  
- filter it with RBAC  
- send only authorized context to the LLM  
- get back a clean natural-language answer

Important for this class:
- for a **salary question**, the notebook reads the requester level directly from `salary_df`
- for a **normal policy question**, the notebook skips the salary-level lookup and still answers from policy context
- the LLM never sees unauthorized salary rows


In [25]:
def call_bedrock(prompt: str, model_id: str = BEDROCK_LLM_MODEL_ID) -> str:
    response = bedrock_runtime.converse(
        modelId=model_id,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": 300, "temperature": 0.1}
    )
    return response["output"]["message"]["content"][0]["text"]

def answer_secure_query(query: str, requester_employee_id: str | None) -> tuple[pd.DataFrame, pd.DataFrame, int | None, str]:
    requester_employee_id = normalize_employee_id(requester_employee_id)
    query_is_salary = is_salary_question(query)

    if query_is_salary:
        requester_level = get_requester_level_from_df(requester_employee_id)
    else:
        requester_level = None

    retrieval_audit_df, authorized_df = build_retrieval_audit_table(
        query=query,
        requester_level=requester_level,
        requester_employee_id=requester_employee_id,
        top_k=TOP_K
    )

    if query_is_salary and not requester_employee_id:
        no_context_answer = (
            "Please provide a requester employee ID for salary questions so RBAC can be applied."
        )
        return retrieval_audit_df, authorized_df, requester_level, no_context_answer

    if len(authorized_df) == 0:
        no_context_answer = (
            "I could not produce an answer because no authorized evidence was available "
            "for this request after retrieval and filtering."
        )
        return retrieval_audit_df, authorized_df, requester_level, no_context_answer

    context = "\n\n".join([
        f"[{i+1}] {t}" for i, t in enumerate(authorized_df["text"].tolist())
    ])

    prompt = f"""
You are an enterprise HR assistant using retrieval-augmented generation (RAG).

Your job:
1. Answer only from the authorized context below.
2. Write the answer in a short, professional, natural style.
3. For salary questions:
   - mention employee id
   - mention monthly salary if present
   - mention annual salary if present
4. For non-salary questions:
   - answer like a normal policy assistant
   - do not mention salary details unless the question is explicitly about salary
5. If the answer is not present in the authorized context, say:
   "I do not know from the authorized context."
6. End with a brief citation line such as: Source: [1] or Source: [1], [2]

Authorized context:
{context}

Question:
{query}

Answer:
""".strip()

    answer = call_bedrock(prompt)
    return retrieval_audit_df, authorized_df, requester_level, answer

sample_retrieval_audit_df, sample_authorized_df, sample_requester_level, sample_secure_answer = answer_secure_query(
    query=query_to_test,
    requester_employee_id=normalized_requester_employee_id
)

sample_requester_level_label = f"L{sample_requester_level}" if sample_requester_level is not None else "Not needed for non-salary question"

print(f"Requester employee id: {normalized_requester_employee_id if normalized_requester_employee_id else '(blank)'}")
print(f"Requester level (read from salary_df only when needed): {sample_requester_level_label}")
print(f"Question: {query_to_test}")

print("\nAuthorized context sent to the LLM:")
display(sample_authorized_df[[col for col in ["doc_type", "chunk_id", "employee_id", "selection_reason", "text"] if col in sample_authorized_df.columns]])

print("\nLLM-generated RAG answer:")
print(sample_secure_answer)


Requester employee id: EMP-1006
Requester level (read from salary_df only when needed): Not needed for non-salary question
Question: What are the rules for remote work?

Authorized context sent to the LLM:


,doc_type,chunk_id,employee_id,selection_reason,text
0,policy,rag-docs/HR-POL-009_Remote_Work_Policy.docx::c...,None,selected by retrieval ranking after RBAC filte...,A combination of on-site and remote work as pe...
1,policy,rag-docs/HR-POL-009_Remote_Work_Policy.docx::c...,None,selected by retrieval ranking after RBAC filte...,"Employees on probation, under Performance Impr..."
2,policy,rag-docs/HR-POL-009_Remote_Work_Policy.docx::c...,None,selected by retrieval ranking after RBAC filte...,10.1 Remote work approval shall be reviewed qu...
3,policy,rag-docs/HR-POL-009_Remote_Work_Policy.docx::c...,None,selected by retrieval ranking after RBAC filte...,This policy applies to all confirmed employees...



LLM-generated RAG answer:
The rules for remote work include a combination of on-site and remote work as per an approved schedule. Employees must use company-approved VPN, endpoint security, and authentication systems for secure access. Remote work approval is reviewed quarterly by HR and the Business Unit Head (BUH). The company reserves the right to revoke remote work privileges with 7 days’ notice for business, security, or performance reasons. Employees on probation, under Performance Improvement Plans (PIP), or under disciplinary review are excluded unless explicitly approved by the Chief Human Resources Officer (CHRO).

Source: [1], [2], [3]


## Step 7 — Request audit summary

This final table is the **request-level audit summary**.

It explains the secure RAG transaction in one row:
- which requester asked the question  
- whether requester level was needed and, if so, what was read from `salary_df`  
- what was asked  
- how many authorized chunks reached the LLM  
- what answer was produced

For non-salary questions, the requester level field clearly shows that no salary-level lookup was needed.


In [26]:
request_audit_summary_df = pd.DataFrame([{
    "audit_purpose": "One-row summary of the secure RAG transaction",
    "requester_employee_id": normalized_requester_employee_id if normalized_requester_employee_id else "(blank)",
    "requester_level_read_from_df": f"L{sample_requester_level}" if sample_requester_level is not None else "Not needed for non-salary question",
    "question": query_to_test,
    "is_salary_question": is_salary_question(query_to_test),
    "requested_employee_ids": ", ".join(extract_requested_employee_ids(query_to_test, normalized_requester_employee_id)),
    "authorized_rows_sent_to_llm": len(sample_authorized_df),
    "authorized_chunk_ids": " | ".join(sample_authorized_df["chunk_id"].tolist()) if len(sample_authorized_df) else None,
    "final_answer_preview": sample_secure_answer[:160] + ("..." if len(sample_secure_answer) > 160 else ""),
    "timestamp_utc": pd.Timestamp.utcnow()
}])

print("Request audit summary: one row per secure RAG run.")
display(request_audit_summary_df)


Request audit summary: one row per secure RAG run.


,audit_purpose,requester_employee_id,requester_level_read_from_df,question,is_salary_question,requested_employee_ids,authorized_rows_sent_to_llm,authorized_chunk_ids,final_answer_preview,timestamp_utc
0,One-row summary of the secure RAG transaction,EMP-1006,Not needed for non-salary question,What are the rules for remote work?,False,,4,rag-docs/HR-POL-009_Remote_Work_Policy.docx::c...,The rules for remote work include a combinatio...,2026-04-19 15:44:17.059668+00:00


## Mini-hack — M2 secure retrieval challenge

Learners can do one of these:
1. Add a new rule for department-based filtering.
2. Block bulk salary questions like “show me everyone’s salaries” for L5.
3. Add a column `decision` with `allowed` or `denied`.
4. Compare fixed chunking vs semantic chunking for policy docs.

## Reflection questions
- Why is relevance alone not enough in enterprise RAG?
- At what layer should RBAC be enforced?
- What evidence would an auditor ask for after a sensitive answer?
